# viva-Mgen: reproducing the Karr 2012 M. genitalium whole-cell model, one study per figure

_Investigation `mgen` — coder reproduction notebook._

**Question.** Can the core cellular processes of the Karr et al. 2012 Mycoplasma genitalium
whole-cell model be re-expressed as native process-bigraph Processes —
composed through shared cell-variable stores — and reproduce the central
quantitative results of each figure of the paper?

A showcase investigation that reproduces the landmark Karr 2012 whole-cell
model of M. genitalium as native process-bigraph processes, with one study
for each of the paper's seven figures. It demonstrates that the model's
central results — 9 h doubling, protein-dominant composition, decoupled
gene expression, emergent cell-cycle regulation, ATP/GTP energy allocation,
and metabolic gene essentiality — are recoverable from a compact, genuinely
mechanistic reimplementation, without the original's MATLAB code, MySQL
knowledge base, or cluster-scale simulation.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-Mgen/viva-Mgen').is_dir():
    REPO = Path('/home/runner/work/viva-Mgen/viva-Mgen')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_mgen.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Fig 1 — Whole-cell integration: six submodels wired through shared cell variables (`fig1-architecture`)

**Question.** Does viva-Mgen actually reproduce Fig 1's central architectural claim — that
the whole-cell model is a set of independent submodels *integrated* through a
shared set of cell variables — as a real, runnable composite rather than a
drawing? Do all six submodels compose, share stores, and run as one integrated
cell?

**Objective.** Build the integrated composite, derive the process↔cell-variable wiring matrix
directly from the composite document (a genuine Fig 1B-style diagram), count
the submodels and the shared/coupled stores, and run the integrated cell to
confirm it composes and stays viable.

**Hypothesis.** Building the fig1_architecture composite should wire exactly six submodels
(metabolism, mass, transcription, translation, rna_decay, protein_decay) to a
common set of bigraph stores such that several cell variables are touched by
more than one submodel (the coupling that makes it "integrated"), and running
the composite for 20 min should keep the cell viable (growth_fraction ≈ 1) and
growing (mass increasing) — reproducing the integrated-architecture picture of
Fig 1A/1B.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig1-architecture ===
STUDY = 'fig1-architecture'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig1-combined**


In [ ]:
# fig1-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig1-a-architecture**


In [ ]:
# fig1-a-architecture
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig1-b-wiring**


In [ ]:
# fig1-b-wiring
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig1-c-dynamics**


In [ ]:
# fig1-c-dynamics
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| integrates-28-submodels | kind=derived_scalar field=n_processes | op range low 28 high 40 provenance {'kind': 'model', 'note': 'Karr 2012 integrates 28 cellular-process submodels (Fig 1A); the viva-Mgen composite wires 30. Test that the integrated cell is composed of ≥28 submodels.'} |
| shared-variable-integration | kind=derived_scalar field=n_stores_coupled | op range low 10 high 90 provenance {'kind': 'model', 'note': 'Karr 2012 integrates the 28 submodels through ~16 shared cell-variable states each 1 s step (Fig 1B). Test that ≥10 cell variables are coupled across ≥2 processes (real integration, not independent modules).'} |
| cell-mass-doubles | kind=derived_scalar field=mass_fold_change | op range low 1.8 high 2.2 provenance {'kind': 'experiment', 'note': 'A dividing cell doubles its mass over one cell cycle (Karr 2012 Fig 2B, τ≈9 h). Test mass fold-change ≈2 over the integrated 9 h run.'} |
| replication-completes | kind=derived_scalar field=replicated_fraction_final | op range low 0.95 high 1.0 provenance {'kind': 'model', 'note': 'The single chromosome is fully replicated once per cycle (Karr 2012 Fig 4B, chromosome copy 1→2). Test replicated_fraction reaches ~1 in the integrated run.'} |
| cell-divides | kind=derived_scalar field=divides | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'The whole-cell model terminates each simulation at cell division, when the septum diameter reaches 0 (Karr 2012; cytokinesis ~1.08 h). Test that the integrated cell divides.'} |
| mrna-produced | kind=derived_scalar field=mrna_species_produced | op range low 1 high 1000 provenance {'kind': 'model', 'note': 'Gene expression is active in the running cell (Karr 2012 Fig 2G). Test that ≥1 mRNA species is produced by the integrated composite.'} |
| protein-produced | kind=derived_scalar field=protein_species_produced | op range low 1 high 1000 provenance {'kind': 'model', 'note': 'Translation is active in the running cell (Karr 2012 Fig 2G/2H). Test that ≥1 protein species is produced by the integrated composite.'} |


## Study: Fig 2 — Cell growth: 9 h doubling time and protein-dominant composition (`fig2-growth`)

**Question.** Does the reduced-but-genuine viva-Mgen cell (FBA metabolism over iPS189
driving mass accumulation) reproduce the trained whole-cell model's core
growth phenotype from Fig 2 — a ~9 h doubling time and a protein-dominant
dry-mass composition?

**Objective.** Run the fig2_growth composite (metabolism + mass) for one cell cycle, fit the
exponential growth rate to recover the doubling time, confirm mass doubles over
the cycle, and read out the dry-mass composition at division.

**Hypothesis.** With growth calibrated to the model's fitted cell-cycle length (32400 s) and
the fitted dry-mass fractions from parameters.json, the integrated cell should
double its mass in ~9 h and show ~62% protein by dry mass, matching Fig 2A/B
(doubling time) and Fig 2C (composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig2-growth ===
STUDY = 'fig2-growth'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig2-combined**


In [ ]:
# fig2-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-a-growth**


In [ ]:
# fig2-a-growth
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-b-composition**


In [ ]:
# fig2-b-composition
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-c-composition-donut**


In [ ]:
# fig2-c-composition-donut
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-d-dynamics**


In [ ]:
# fig2-d-dynamics
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-g-expression**


In [ ]:
# fig2-g-expression
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig2-h-mrna-protein**


In [ ]:
# fig2-h-mrna-protein
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| doubling-time-near-9h | kind=derived_scalar field=doubling_time_h | op range low 8.3 high 9.5 provenance {'kind': 'experiment', 'note': 'M. genitalium doubling time τ ≈ 9.0 h measured (Fig 2A); the simulated median is 8.9 h (Fig 2B). Test the exponential-fit doubling time falls in [8.3, 9.5] h.'} |
| mass-doubles-over-cycle | kind=derived_scalar field=final_mass_ratio | op range low 1.8 high 2.2 provenance {'kind': 'model', 'note': 'A cell doubles its mass between birth and division (Karr 2012 Fig 2B). Test mass fold-change ≈2 over the 9 h cycle.'} |
| protein-dominant-composition | kind=derived_scalar field=protein_fraction | op range low 0.62 high 0.75 provenance {'kind': 'experiment', 'note': 'Dry-mass composition is protein-dominant: protein ≈62–75% (Karr 2012 Fig 2C vs Morowitz 1962). Test the protein dry-mass fraction is in [0.62, 0.75].'} |
| rna-fraction-small | kind=derived_scalar field=rna_fraction | op range low 0.05 high 0.15 provenance {'kind': 'experiment', 'note': 'RNA is ≈9% of dry mass (Karr 2012 Fig 2C / Morowitz 1962). Test the RNA dry-mass fraction is in [0.05, 0.15].'} |
| dna-doubles-in-s-phase | kind=derived_scalar field=dna_fold_change | op range low 1.8 high 2.2 provenance {'kind': 'model', 'note': 'Single-cell DNA content steps from 1× to 2× during S-phase (Karr 2012 Fig 2D). Test DNA fold-change ≈2 over the cycle.'} |
| mrna-low-copy-bursty | kind=derived_scalar field=mean_mrna_per_gene | op range low 0.0 high 2.5 provenance {'kind': 'experiment', 'note': 'mRNA is present at low, discrete copy numbers (0–2 per gene) due to short half-lives and bursty synthesis (Karr 2012 Fig 2G). Test mean mRNA per gene ≤2.5.'} |
| mrna-protein-decoupled | kind=derived_scalar field=mrna_protein_abs_corr | op range low 0.0 high 0.4 provenance {'kind': 'experiment', 'note': 'Single-cell mRNA (transient, bursty) and protein (cumulative, long-lived) copy numbers are decoupled — no correlation across a population (Karr 2012 Fig 2H). Test |Pearson r| < 0.4.'} |


## Study: Fig 3 — Chromosome DNA-protein interactions (viva-Mgen) (`fig3-expression`)

**Question.** Does the reduced viva-Mgen expression module (stochastic transcription +
translation + RNA/protein decay on a representative gene panel) reproduce the
qualitative single-cell gene-expression dynamics of Fig 3 / 2G-2H — bursty,
low-copy mRNA alongside protein that accumulates to much higher, more stable
copy numbers (the mRNA↔protein decoupling)?

**Objective.** Run the fig3_expression composite (transcription + translation + rna_decay +
protein_decay) for one hour at the default 1 s interval, gather per-gene mRNA
and protein counts, and read out mean mRNA per gene, final total protein, the
protein-to-mRNA ratio, and the fraction of panel genes expressed.

**Hypothesis.** With Poisson synthesis and Poisson decay running through shared stores, mRNA
should stay low and bursty (order 0-2 copies per gene, set by short half-lives
and low synthesis rates), while protein — synthesized proportionally to mRNA
and decaying far more slowly — should accumulate to copy numbers orders of
magnitude above mRNA, reproducing the Fig 2H decoupling qualitatively.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | seed=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig3-expression ===
STUDY = 'fig3-expression'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig3-combined**


In [ ]:
# fig3-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-a-occupancy**


In [ ]:
# fig3-a-occupancy
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-b-explored**


In [ ]:
# fig3-b-explored
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-c-rna-expressed**


In [ ]:
# fig3-c-rna-expressed
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-d-polymerase-traces**


In [ ]:
# fig3-d-polymerase-traces
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-e-collision-matrix**


In [ ]:
# fig3-e-collision-matrix
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig3-f-collisions-vs-density**


In [ ]:
# fig3-f-collisions-vs-density
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| chromosome-50pct-bound-by-6min | kind=derived_scalar field=pct_explored_at_6min | op range low 40 high 65 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3B: 50% of the chromosome is bound by ≥1 protein within the first 6 min.'} |
| chromosome-90pct-bound-by-20min | kind=derived_scalar field=pct_explored_at_20min | op range low 80 high 100 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3B: 90% of the chromosome bound within 20 min.'} |
| rnap-binds-90pct-within-49min | kind=derived_scalar field=rnap_90pct_time_min | op range low 0 high 49 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3B: RNA polymerase binds 90% of the chromosome within the first 49 min.'} |
| rna-expression-t50-18min | kind=derived_scalar field=rna_t50_min | op range low 10 high 30 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3C: RNA-expression t50 = 18 min.'} |
| rna-expression-90pct-by-143min | kind=derived_scalar field=rna_t90_min | op range low 60 high 200 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3C: 90% of genes expressed within the first 143 min.'} |
| collisions-per-cycle-gt-30000 | kind=derived_scalar field=n_collisions_per_cycle | op range low 30000 high 1000000000.0 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3F: >30,000 collisions per cell (~0.93 displacements/s).'} |
| collisions-mostly-by-rnap | kind=derived_scalar field=frac_collisions_by_rnap | op range low 0.7 high 1.0 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3F: collisions caused mostly by RNA polymerase (84%) and DNA pol (8%).'} |
| collisions-displace-smc | kind=derived_scalar field=frac_collisions_displacing_smc | op range low 0.5 high 0.85 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3F: displaced proteins are mostly SMC (70%) and SSB (6%).'} |
| collisions-density-positive | kind=derived_scalar field=collisions_density_pearson_r | op range low 0.2 high 1.0 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 3F: collisions vs DNA-bound protein density are positively correlated.'} |


## Study: Fig 4 — Emergent, unregulated control of cell-cycle duration (`fig4-cell-cycle`)

**Question.** Does the viva-Mgen replication submodel reproduce Fig 4's central finding —
that M. genitalium's cell-cycle duration is controlled *emergently*, without a
dedicated genetic regulator, through the coupling of replication initiation and
a dNTP surplus that it builds up?

**Objective.** Simulate a population of ~128 single cells with random birth DnaA and dNTP
levels by stepping the ReplicationReproductionProcess directly, recover the
three Fig 4 single-cell correlations, and render a representative single-cell
dynamics trajectory through the fig4_cell_cycle composite.

**Hypothesis.** If DnaA accumulates stochastically to trigger initiation while dNTPs
accumulate (unconsumed) during that same initiation phase, then across single
cells (1) more birth DnaA shortens initiation (Fig 4C), (2) a larger dNTP pool
at replication start shortens replication (Fig 4D), and (3) initiation and
replication durations are inversely correlated (Fig 4E) — a longer initiation
buys a bigger dNTP surplus that speeds replication, buffering total cycle length.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | initial_dnaA=5, initial_dntp=0, seed=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig4-cell-cycle ===
STUDY = 'fig4-cell-cycle'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig4-combined**


In [ ]:
# fig4-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig4-a-phase-durations**


In [ ]:
# fig4-a-phase-durations
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig4-b-single-cell-dynamics**


In [ ]:
# fig4-b-single-cell-dynamics
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig4-c-dnaA-vs-init**


In [ ]:
# fig4-c-dnaA-vs-init
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig4-d-dntp-vs-repl**


In [ ]:
# fig4-d-dntp-vs-repl
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig4-e-init-vs-repl**


In [ ]:
# fig4-e-init-vs-repl
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-cycle-duration-9h | kind=derived_scalar field=cell_cycle_h | op range low 8.5 high 9.5 provenance {'kind': 'experiment', 'note': 'Karr 2012: 9.0 h cell cycle (median in-silico doubling 8.9 h; Fig 2B/4A).'} |
| initiation-duration-3.6h | kind=derived_scalar field=init_dur_h | op range low 3.0 high 4.2 provenance {'kind': 'model', 'note': 'Karr 2012 Time state: replicationInitiationDuration = 12960 s ≈ 3.6 h.'} |
| replication-duration-4.33h | kind=derived_scalar field=repl_dur_h | op range low 3.8 high 4.9 provenance {'kind': 'model', 'note': 'Karr 2012 Time state: replicationDuration = 15571 s ≈ 4.33 h.'} |
| cytokinesis-duration-1.08h | kind=derived_scalar field=cyto_dur_h | op range low 0.9 high 1.3 provenance {'kind': 'model', 'note': 'Karr 2012 Time state: cytokinesisDuration = 3869 s ≈ 1.08 h.'} |
| replication-duration-variable | kind=derived_scalar field=cv_repl_pct | op range low 25 high 55 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 4A: replication-duration CV = 38.5% across 128 cells.'} |
| total-cycle-less-variable-than-phases | kind=derived_scalar field=total_cv_below_phase_cvs | op range low 1.0 high 1.0 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 4A: cycle CV 9.4% < initiation 64.3% and replication 38.5% — emergent buffering.'} |
| initiation-replication-inversely-correlated | kind=derived_scalar field=r_init_repl | op range low -1.0 high -0.7 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 4E: longer initiation → shorter replication (dNTP-surplus buffering).'} |
| more-dnaa-shortens-initiation | kind=derived_scalar field=r_dnaA_init | op range low -1.0 high -0.35 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 4C: initial DnaA vs initiation duration, R² ≈ 0.49 (r ≈ -0.7).'} |
| dntp-controls-replication-duration | kind=derived_scalar field=r_dntp_repl | op range low -1.0 high -0.7 provenance {'kind': 'experiment', 'note': 'Karr 2012 Fig 4D: higher dNTP at replication start → shorter replication.'} |


## Study: Fig 5 — Global distribution of cellular energy: ATP > GTP synthesis and a translation-dominated budget (`fig5-energy`)

**Question.** Does the reduced viva-Mgen cell reproduce Fig 5's global energy picture — that
ATP and GTP are the dominant synthesized carriers with ATP > GTP (Fig 5A), and
that the cellular energy budget is dominated by translation, ahead of
transcription (Fig 5D)?

**Objective.** Run the fig5_energy composite for one hour (metabolism/mass at 60 s,
expression at 1 s) to time-average ATP/GTP production, then step the
transcription and translation processes directly for one hour to tally their
NTP/GTP consumption, and compute the modeled energy shares and the ATP:GTP
synthesis ratio.

**Hypothesis.** FBA over iPS189 should carry far more flux through ATP synthase (ATPS4r) than
through the GTP-producing kinases, giving ATP > GTP > 0. And because each
peptide bond costs ~2 GTP and translation events vastly outnumber transcription
events (mRNA accumulates and each transcript is translated many times over),
the modeled expression energy should be overwhelmingly translation, matching
the ordering translation >> transcription in Fig 5D.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig5-energy ===
STUDY = 'fig5-energy'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig5-combined**


In [ ]:
# fig5-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig5-a-synthesis**


In [ ]:
# fig5-a-synthesis
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig5-b-population**


In [ ]:
# fig5-b-population
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig5-c-by-process**


In [ ]:
# fig5-c-by-process
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig5-d-allocation**


In [ ]:
# fig5-d-allocation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| atp-exceeds-gtp-synthesis | kind=derived_scalar field=atp_to_gtp_ratio | op range low 2.0 high 8.0 provenance {'kind': 'model', 'note': 'Fig 5A: ATP and GTP are the dominant synthesized carriers with ATP > GTP; iPS189 FBA gives ~4.0.'} |
| atp-gtp-dominate-redox | kind=derived_scalar field=atp_gtp_to_redox_ratio | op range low 1000.0 high 1000000000000.0 provenance {'kind': 'experiment', 'note': 'Fig 5A: ATP/GTP are synthesized >1000x more than NAD(H)/NADP(H)/FAD(H2). Measured as total ATP+GTP production flux vs de-novo carrier biosynthesis flux (NADS1/NADK/FMNATr) — the carriers are recycled but their molecules are made de novo only rarely; iPS189 FBA gives ~1600x.'} |
| translation-largest-energy-consumer | kind=derived_scalar field=translation_energy_share | op range low 0.2 high 0.4 provenance {'kind': 'experiment', 'note': 'Fig 5D: Translation = 29.0% of average ATP+GTP usage.'} |
| aminoacylation-energy-share | kind=derived_scalar field=aminoacylation_energy_share | op range low 0.1 high 0.22 provenance {'kind': 'experiment', 'note': 'Fig 5D: tRNA aminoacylation = 15.1% of ATP+GTP usage.'} |
| transcription-energy-share | kind=derived_scalar field=transcription_energy_share | op range low 0.04 high 0.12 provenance {'kind': 'experiment', 'note': 'Fig 5D: Transcription = 7.1% of ATP+GTP usage. (Reduced-panel transcription accounting currently under-counts NTP relative to translation.)'} |
| unaccounted-energy-gap | kind=derived_scalar field=unaccounted_share | op range low 0.35 high 0.5 provenance {'kind': 'model', 'note': 'Fig 5D: 44.3% of experimentally-observed energy production is unaccounted for by the model — a genuine reported discrepancy.'} |
| ntp-use-invariant-across-cells | kind=derived_scalar field=ntp_use_cv_across_cells | op range low 0.0 high 0.15 provenance {'kind': 'model', 'note': 'Fig 5B: total ATP/GTP use is nearly invariant across cells except the very slowest — metabolism (not expression) sets cycle length.'} |


## Study: Fig 6A — Single-gene-disruption essentiality: model vs experiment (`fig6-gene-essentiality`)

**Question.** Does the viva-Mgen FBA metabolism submodel reproduce Fig 6A — the
single-gene-disruption essentiality phenotype — for the metabolic genes of
M. genitalium: does an in-silico single-gene knockout predict which genes are
essential, matching the reference essentiality calls?

**Objective.** For every metabolic gene with a reference call, run a single-gene FBA knockout
via MetabolismFbaReproductionProcess, call it essential when growth_fraction
drops below 0.05, cross-tabulate against the reference calls into a 2x2
confusion matrix, and score accuracy, sensitivity, and specificity. Also run
the fig6_gene_essentiality composite once for a representative essential gene
(MG_023) to record a canonical baseline and confirm growth collapses.

**Hypothesis.** For the ~125 genes the iPS189 metabolic reconstruction covers (the paper notes
metabolic-gene disruptions are the most debilitating), a single-gene FBA
knockout that collapses growth should mark the gene essential, and the
model-vs-reference confusion matrix should land near the paper's whole-cell
accuracy (~79% overall; ~87% for the iPS189 metabolic model on essentiality).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig6-gene-essentiality ===
STUDY = 'fig6-gene-essentiality'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig6-combined**


In [ ]:
# fig6-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig6-a-confusion**


In [ ]:
# fig6-a-confusion
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig6-b-phenotype-classes**


In [ ]:
# fig6-b-phenotype-classes
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig6-c-growthfraction-dist**


In [ ]:
# fig6-c-growthfraction-dist
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig6-d-performance-summary**


In [ ]:
# fig6-d-performance-summary
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**confusion-matrix**


In [ ]:
# confusion-matrix
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**accuracy-summary**


In [ ]:
# accuracy-summary
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig6-b-pathologies**


In [ ]:
# fig6-b-pathologies
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| essentiality-accuracy | kind=derived_scalar field=essentiality_accuracy | op range low 0.78 high 1.0 provenance {'kind': 'experiment', 'note': 'Fig 6A: 79% accuracy (316/401 correct) vs Glass et al. 2006, p<1e-7. (Metabolic-subset FBA currently reaches ~76%.)'} |
| essentiality-sensitivity | kind=derived_scalar field=sensitivity | op range low 0.75 high 1.0 provenance {'kind': 'experiment', 'note': 'Fig 6A confusion table (Glass 2006): sensitivity ~0.79.'} |
| essentiality-specificity | kind=derived_scalar field=specificity | op range low 0.65 high 1.0 provenance {'kind': 'experiment', 'note': 'Fig 6A confusion table (Glass 2006): specificity ~0.77.'} |
| growth-distribution-bimodal | kind=derived_scalar field=growth_distribution_bimodal | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6/S2: single-gene-KO growth is bimodal — genes are essential (growth~0) or non-essential (growth~1), few intermediate.'} |
| pathology-metabolic-non-growing | kind=derived_scalar field=pathology_metabolic_non_growing | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6B: metabolic-gene disruptions are the most debilitating — non-growing.'} |
| pathology-rna-ko-stops-rna | kind=derived_scalar field=pathology_rna_ko_stops_rna | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6B: RNA-synthesis disruptions (rpoE) fail to accumulate RNA (and cascade to protein).'} |
| pathology-protein-ko-stops-protein | kind=derived_scalar field=pathology_protein_ko_stops_protein | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6B: protein-synthesis disruptions (asnS) fail to accumulate protein.'} |
| pathology-dna-ko-non-replicative | kind=derived_scalar field=pathology_dna_ko_non_replicative | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6B: DNA-synthesis disruptions (dnaN) never double their DNA.'} |
| pathology-cytokinesis-ko-non-fissive | kind=derived_scalar field=pathology_cytokinesis_ko_non_fissive | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 6B: cytokinesis disruptions (parC) never close the septum / divide.'} |


## Study: Fig 7E-G — Kinetic parameters: growth depends sigmoidally on an enzyme's kcat (Vmax proxy) (`fig7-kinetic-parameters`)

**Question.** Does the reduced-but-genuine viva-Mgen cell reproduce Fig 7E-G's central
quantitative claim — that predicted growth rate depends on an enzyme's kinetic
parameter (kcat/Vmax), producing a SIGMOIDAL growth-vs-kcat curve that rises
from near-zero at low activity and saturates at the wild-type rate once the
enzyme is no longer rate-limiting?

**Objective.** Programmatically identify a growth-limiting reaction in the iPS189 network
(one whose bound binds when throttled yet whose growth recovers to the
wild-type plateau before the top of the sweep), sweep its flux bound over
np.logspace(-2, 1, 16) with MetabolismFbaReproductionProcess, and read out
whether the kcat→growth curve is monotonic, saturating, and has a clear
dynamic range.

**Hypothesis.** If a genuinely growth-limiting reaction's FBA flux bound is used as a kcat
proxy and swept over ~three decades (0.01–10x wild-type), growth_fraction
should rise monotonically with the bound and then plateau at 1.0, mirroring the
saturating kcat→growth dependence the paper uses to reconcile model and
experiment for lpdA/deoD/thyA.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig7-kinetic-parameters ===
STUDY = 'fig7-kinetic-parameters'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig7-combined**


In [ ]:
# fig7-combined
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig7-a-ko-scatter**


In [ ]:
# fig7-a-ko-scatter
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig7-b-kcat-sigmoid**


In [ ]:
# fig7-b-kcat-sigmoid
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig7-c-kcat-overlay**


In [ ]:
# fig7-c-kcat-overlay
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**fig7-d-dynamic-range**


In [ ]:
# fig7-d-dynamic-range
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**kcat-growth-curve**


In [ ]:
# kcat-growth-curve
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| growth-monotonic-in-kcat | kind=derived_scalar field=growth_is_monotonic_in_kcat | op range low 1.0 high 1.0 provenance {'kind': 'theory', 'note': 'Fig 7E: relaxing a rate-limiting Vmax cannot decrease FBA growth.'} |
| growth-saturates-at-wt | kind=derived_scalar field=growth_saturates_at_wt | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 7E: the predicted-growth-vs-kcat curve saturates at the wild-type growth rate.'} |
| growth-sigmoidal-in-kcat | kind=derived_scalar field=growth_is_sigmoidal | op range low 1.0 high 1.0 provenance {'kind': 'model', 'note': 'Fig 7E: growth depends sigmoidally on the enzyme kcat.'} |
| growth-dynamic-range | kind=derived_scalar field=growth_dynamic_range | op range low 0.7 high 1.0 provenance {'kind': 'model', 'note': 'Fig 7E: reducing kcat drives growth from the WT rate down toward non-growing.'} |
| growth-at-max-kcat-near-wt | kind=derived_scalar field=growth_at_max_kcat | op range low 0.95 high 1.05 provenance {'kind': 'model', 'note': 'Fig 7E: high kcat -> WT growth (enzyme no longer limiting).'} |
| growth-at-min-kcat-low | kind=derived_scalar field=growth_at_min_kcat | op range low 0.0 high 0.3 provenance {'kind': 'model', 'note': 'Fig 7E: at low kcat the enzyme is strongly rate-limiting; ~5% throughput reaches WT (nox/lpdA).'} |
| kcat-half-max-below-wt | kind=derived_scalar field=kcat_half_max | op range low 0.001 high 0.5 provenance {'kind': 'model', 'note': 'Fig 7E: growth is half-maximal at a small fraction of WT kcat, then saturates.'} |
